# Second Draft — redraw a room, free

Upload one photograph of a room and get several redesigned versions back,
with the doors, windows and walkways left exactly where they are.

**No API keys.** Everything runs inside this Colab runtime on open weights.

| Pass | Model |
|---|---|
| Read + name the regions | SegFormer, fine-tuned on ADE20K |
| Hold structure | the repo's own lock policy |
| Redraw | Stable Diffusion inpainting, via `diffusers` |

**Before you run anything:** switch on the GPU, or this will take ~15 minutes
per image instead of seconds. `Runtime → Change runtime type → T4 GPU`.

Then run the cells in order.

In [ ]:
#@title 1 · Check the GPU is on
import torch

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Ready — a room will take under a minute.')
else:
    print('NO GPU. This will still work, but each option takes several minutes.')
    print('Runtime -> Change runtime type -> T4 GPU, then run this cell again.')

In [ ]:
#@title 2 · Get the code and the libraries
!git clone --quiet --branch claude/interior-design-room-analysis-yj4fvg https://github.com/manuaishika/interior-design 2>/dev/null || echo 'already cloned'
%cd /content/interior-design
!git pull --quiet 2>/dev/null

# torch ships with Colab; these are the rest. Deliberately not installing
# replicate or openai — the keyless backend never imports them.
!pip install --quiet diffusers transformers accelerate pydantic-settings

import sys
sys.path.insert(0, '/content/interior-design')
print('\nReady.')

In [ ]:
#@title 3 · Upload a room photograph
from google.colab import files
from IPython.display import display
from PIL import Image
import io

uploaded = files.upload()
if not uploaded:
    raise SystemExit('No file chosen — run this cell again.')

ROOM_NAME = list(uploaded)[0]
ROOM_BYTES = uploaded[ROOM_NAME]

print(f'{ROOM_NAME} — {len(ROOM_BYTES) / 1e6:.1f} MB')
display(Image.open(io.BytesIO(ROOM_BYTES)).copy().resize((480, 270)))

In [ ]:
#@title 4 · Choose the direction
STYLE = "japandi"  #@param ["scandinavian", "mid-century-modern", "industrial", "japandi", "bohemian", "modern-luxury"]
OPTIONS = 3  #@param {type:"slider", min:1, max:4, step:1}
DEPTH = "renovate"  #@param ["renovate", "restyle"]
ANYTHING_ELSE = ""  #@param {type:"string"}

print(f'{OPTIONS} option(s), {STYLE}, {DEPTH}')

In [ ]:
#@title 5 · Redraw the room
#
# First run downloads ~2.5 GB of weights and takes a few minutes.
# Every run after that is fast.
import logging, time
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)

from app.config import Settings
from app.pipeline import run_pipeline

settings = Settings(
    backend='local',          # no keys, no hosted calls
    max_image_edge=768,       # analysis resolution
    generation_steps=30,
)

started = time.time()
analysis, generations = await run_pipeline(
    ROOM_BYTES, STYLE, settings,
    extra_prompt=ANYTHING_ELSE,
    variants=OPTIONS,
    profile=DEPTH,
)
print(f'\nDone in {time.time() - started:.0f}s — {len(generations)} option(s).')

In [ ]:
#@title 6 · What the system found in your room
held = [o for o in analysis.objects if o.locked]
open_ = [o for o in analysis.objects if not o.locked]

print(f'{len(analysis.objects)} regions · {len(held)} held · {len(open_)} redrawn\n')
print(f"{'region':<22}{'treatment':<12}{'box'}")
print('-' * 62)
for o in analysis.objects:
    b = o.bounding_box
    mark = 'HELD' if o.locked else 'redrawn'
    print(f'{o.label:<22}{mark:<12}{b.x},{b.y} {b.width}x{b.height}')

if not held:
    print('\nNothing was held. No door, window or walkway was recognised —')
    print('try a photograph where a doorway or window is clearly visible.')

In [ ]:
#@title 7 · The versions
import base64, io
from PIL import Image
import matplotlib.pyplot as plt

def decode(b64):
    return Image.open(io.BytesIO(base64.b64decode(b64)))

panels = [(Image.open(io.BytesIO(ROOM_BYTES)).convert('RGB'), 'As photographed')]
panels += [(decode(g.image_base64), f'Option {i + 1} · seed {g.seed}')
           for i, g in enumerate(generations)]
panels.append((decode(generations[0].inpaint_mask_base64),
               'The boundary\nwhite redrawn · black held'))

cols = min(len(panels), 3)
rows = (len(panels) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
for ax, (img, title) in zip(list(axes.flat) if rows * cols > 1 else [axes], panels):
    ax.imshow(img); ax.set_title(title, fontsize=11); ax.axis('off')
for ax in list(axes.flat)[len(panels):] if rows * cols > 1 else []:
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
#@title 8 · Save everything
import json, pathlib

out = pathlib.Path('/content/second-draft-output')
out.mkdir(exist_ok=True)

for i, g in enumerate(generations):
    decode(g.image_base64).save(out / f'option_{i + 1}.png')
decode(generations[0].inpaint_mask_base64).save(out / 'boundary_mask.png')
(out / 'analysis.json').write_text(json.dumps(analysis.model_dump(), indent=2))

for p in sorted(out.iterdir()):
    print(p.name)

# Uncomment to pull them onto your machine:
# !zip -qr /content/second-draft-output.zip /content/second-draft-output
# files.download('/content/second-draft-output.zip')

---

### If the options look too similar

They differ only by seed. Re-run cell 5 — a fresh seed set is drawn each time.

### If the room's shape drifts

`renovate` leaves most of the frame open, and diffusion over a large mask can
lose the perspective. Set `DEPTH = "restyle"` in cell 4 to hold more of the
room, or raise `generation_steps` in cell 5.

### If nothing was held

The segmentation model has to actually see a door or a window. A photograph
taken from a corner, showing at least one opening, works much better than a
tight shot of one wall.